# Phase 8 — LLM Checks

Validates the `LLMModel` layer before wiring into `VerifyClaimService` (Phase 11).

Goals:
- Load the active LLM (dev: Qwen 1.5B / prod: Llama 3.1 8B) and confirm it generates valid JSON
- Inspect the loaded system prompt
- Run 3 sanity claims: supported, refuted, noisy/mixed
- Verify per-source classification table (NLI hint → LLM class, rationale)
- Assert JSON validity for all 3 claims
- Assert at least 1 `correlated_context` in the noisy claim
- Final checklist

**Model tiers** (change `LLM_MODEL_NAME` in `constants.py` to switch):
- Dev  → `LLM_DEV_MODEL_NAME`  = Qwen/Qwen2.5-1.5B-Instruct  (~3GB, ~10s inference on MPS)
- Prod → `LLM_PROD_MODEL_NAME` = meta-llama/Llama-3.1-8B-Instruct (~16GB, ~1min on MPS)

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

from backend.app.models.llm_model import LLMModel, LLMResult, SourceClassification, get_llm_model
from backend.app.models.nli_model import NLIModel
from backend.app.services.stance_service import StanceService
from backend.app.services.cache_service import CacheService
from backend.app.services.retrieval_service import RetrievalService
from backend.app.services.ranking_service import RankingService
from backend.app.services.evidence_expansion_service import EvidenceExpansionService
from backend.app.retrieval.retriever_registry import RetrieverRegistry
from backend.app.retrieval.wikipedia_retriever import WikipediaRetriever
from backend.app.retrieval.factcheck_retriever import FactCheckRetriever
from backend.app.retrieval.guardian_retriever import GuardianRetriever
from backend.app.retrieval.newsapi_retriever import NewsApiRetriever
from backend.app.retrieval.gdelt_retriever import GDELTRetriever
from backend.app.retrieval.livewiki_retriever import LiveWikiRetriever
from backend.app.utils.constants import (
    LLM_MODEL_NAME,
    LLM_DEV_MODEL_NAME,
    LLM_PROD_MODEL_NAME,
    LLM_FALLBACK_MODEL_NAME,
    LLM_DEVICE,
    LLM_MAX_INPUT_SOURCES,
    LLM_PROMPT_VERSION,
    NLI_MODEL_NAME,
    NLI_CONFIRM_MODEL_NAME,
    DEFAULT_RETRIEVAL_CACHE_DIR,
    DEFAULT_PROCESSED_DIR,
    FEVER_EVIDENCE_SNIPPETS_OUTPUT_FILENAME,
)

CACHE_DIR = PROJECT_ROOT / DEFAULT_RETRIEVAL_CACHE_DIR

tier = "DEV" if LLM_MODEL_NAME == LLM_DEV_MODEL_NAME else "PROD"
print(f"Active tier     : {tier}")
print(f"LLM model       : {LLM_MODEL_NAME}")
print(f"Fallback model  : {LLM_FALLBACK_MODEL_NAME}")
print(f"LLM device      : {LLM_DEVICE}")
print(f"Prompt version  : {LLM_PROMPT_VERSION}")
print(f"Max LLM sources : {LLM_MAX_INPUT_SOURCES}")
print(f"Cache dir       : {CACHE_DIR}")

/Users/vraj21/Desktop/Projects/Fact Checker/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Active tier     : DEV
LLM model       : Qwen/Qwen2.5-1.5B-Instruct
Fallback model  : Qwen/Qwen2.5-7B-Instruct
LLM device      : mps
Prompt version  : v1
Max LLM sources : 5
Cache dir       : /Users/vraj21/Desktop/Projects/Fact Checker/data/cache/retrieval_results


## Prompt Inspection

In [2]:
prompt_path = PROJECT_ROOT / "backend" / "app" / "prompts" / f"classify_sources_{LLM_PROMPT_VERSION}.txt"
system_prompt = prompt_path.read_text(encoding="utf-8").strip()

print(f"Prompt file : {prompt_path}")
print(f"Length      : {len(system_prompt)} chars")
print("\n" + "=" * 60)
print(system_prompt)
print("=" * 60)

Prompt file : /Users/vraj21/Desktop/Projects/Fact Checker/backend/app/prompts/classify_sources_v1.txt
Length      : 1380 chars

You are a fact-checking assistant. You will be given a CLAIM and a list of SOURCES with short evidence snippets.

Your task:
1. For each source, classify it using ONLY the snippet text provided. Do NOT use external knowledge.
2. Produce a one-sentence rationale grounded in the snippet.
3. Identify the single best source for evaluating this claim.
4. Rank up to 5 sources by usefulness (most useful first).
5. Give an overall verdict for the claim based on the evidence.

Classification options:
- direct_support    : The snippet explicitly and directly confirms the claim is true.
- direct_refute     : The snippet explicitly and directly contradicts the claim.
- correlated_context: The snippet is topically related but does not directly prove or disprove the claim.
- insufficient      : The snippet has no meaningful relevance to the claim.

Overall verdict options: 

## Load Models

In [3]:
print("Loading NLI fast model...")
nli_fast = NLIModel(model_name=NLI_MODEL_NAME)

print("Loading NLI confirm model...")
nli_confirm = NLIModel(model_name=NLI_CONFIRM_MODEL_NAME)

# get_llm_model() returns a singleton — re-running this cell does NOT reload the model
print(f"Loading LLM ({LLM_MODEL_NAME})...")
print("  Re-running this cell is instant — singleton reuses the already-loaded model.")
llm = get_llm_model(model_name=LLM_MODEL_NAME, device=LLM_DEVICE, fallback_model_name=LLM_FALLBACK_MODEL_NAME)

print("\nAll models loaded.")

Loading NLI fast model...


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 1746.07it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading NLI confirm model...


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 801.72it/s, Materializing param=pooler.dense.weight]                                      
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading LLM (Qwen/Qwen2.5-1.5B-Instruct)...
  Re-running this cell is instant — singleton reuses the already-loaded model.
[LLM] Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
      (First run downloads model weights — cached permanently after that)
[LLM] Tokenizer ready.
[LLM] Loading model weights (dtype=torch.float16, device=mps)...
      This takes 3-10 min on first load (cached after that).


Loading weights: 100%|██████████| 338/338 [00:02<00:00, 134.54it/s, Materializing param=model.norm.weight]                              


[LLM] Model weights loaded.
[LLM] Active model: Qwen/Qwen2.5-1.5B-Instruct

All models loaded.


## Build Retrieval Pipeline

In [4]:
registry = RetrieverRegistry()
registry.register(WikipediaRetriever(
    evidence_snippets_path=PROJECT_ROOT / DEFAULT_PROCESSED_DIR / FEVER_EVIDENCE_SNIPPETS_OUTPUT_FILENAME
))
registry.register(FactCheckRetriever(api_key=os.getenv("FACTCHECK_API_KEY")))
registry.register(GuardianRetriever(api_key=os.getenv("GUARDIAN_API_KEY")))
registry.register(NewsApiRetriever(api_key=os.getenv("NEWSAPI_KEY")))
registry.register(GDELTRetriever())
registry.register(LiveWikiRetriever())

cache = CacheService(CACHE_DIR)
retrieval_svc = RetrievalService(
    registry=registry,
    cache=cache,
    ranking=RankingService(),
    expansion=EvidenceExpansionService(),
)

stance_svc = StanceService(
    model=nli_fast,
    cache=cache,
    confirm_model=nli_confirm,
)

print("Retrieval pipeline ready (cascade NLI enabled).")

Retrieval pipeline ready (cascade NLI enabled).


## Helper: run full pipeline on a claim

In [ ]:
def run_claim(claim: str, label: str) -> LLMResult:
    """Retrieve → NLI → LLM for a single claim. Prints a summary table."""
    print(f"\n{'='*70}")
    print(f"[{label.upper()}] {claim}")
    print(f"{'='*70}")

    sources = retrieval_svc.retrieve(claim, max_results=10, use_cache=False)
    print(f"  Retrieved : {len(sources)} sources")

    if not sources:
        print("  [WARN] No sources — skipping LLM.")
        return None

    classified, _nli_results = stance_svc.classify(claim, sources)
    llm_input = classified[:LLM_MAX_INPUT_SOURCES]
    print(f"  NLI done  : {len(llm_input)} sources sent to LLM")

    result = llm.classify(claim, llm_input)

    # Print table
    class_by_idx = {sc.index: sc for sc in result.sources}
    print(f"\n  {'i':<4} {'llm_class':<22} {'nli_hint':<14} {'type':<12} title")
    print(f"  {'-'*75}")
    for i, src in enumerate(llm_input, start=1):
        sc = class_by_idx.get(i)
        classification = sc.classification if sc else "insufficient"
        rationale = sc.rationale if sc else ""
        nli = src.stance_hint or "none"
        title = src.title[:40] if src.title else ""
        print(f"  [{i}]  {classification:<22} {nli:<14} {src.source_type:<12} {title}")
        if rationale:
            print(f"        rationale: {rationale[:90]}")

    print(f"\n  verdict     : {result.overall_verdict}")
    print(f"  confidence  : {result.confidence:.2f}")
    print(f"  best source : [{result.best_source_index}]")
    print(f"  explanation : {result.short_explanation}")

    return result

## 3 Sanity Claims

In [6]:
# Claim 1 — should be SUPPORTED (clear Wikipedia evidence)
result_supported = run_claim(
    "The Eiffel Tower is located in Paris, France.",
    label="supported",
)


[SUPPORTED] The Eiffel Tower is located in Paris, France.
  Retrieved : 7 sources
  NLI done  : 5 sources sent to LLM
[LLM] Generating... (prompt=784 tokens, max_new=256)


LLMModel: first parse attempt failed (JSON decode error: Expecting ',' delimiter: line 6 column 198 (char 819)), retrying.


[LLM] Generation complete.
[LLM] Generating... (prompt=806 tokens, max_new=256)
[LLM] Generation complete.


ValueError: LLMModel: failed to parse JSON response after retry. Last error: JSON decode error: Expecting ',' delimiter: line 6 column 159 (char 768)
Raw output: ```json
{
  "sources": [
    {"index": 2, "classification": "direct_support", "rationale": "The snippet states that the Champ de Mars is a public greenspace in Paris, France, which includes the Eiffel Tower."},
    {"index": 3, "classification": "direct_support", "rationale": "The snippet directly mentions that the Eiffel Tower is located in the Champ de Mars, which is in Paris, France."},
    {"index": 4, "classification": "direct_support", "rationale": "The snippet indicates that the Eiffel To

In [7]:
# Claim 2 — should be REFUTED or MIXED (Great Wall myth)
result_refuted = run_claim(
    "The Great Wall of China is visible from space with the naked eye.",
    label="refuted",
)


[REFUTED] The Great Wall of China is visible from space with the naked eye.
  Retrieved : 6 sources
  NLI done  : 5 sources sent to LLM
[LLM] Generating... (prompt=814 tokens, max_new=256)
[LLM] Generation complete.

  i    llm_class              nli_hint       type         title
  ---------------------------------------------------------------------------
  [1]  insufficient           insufficient   guardian     Donald Trump is making China great again
  [2]  insufficient           insufficient   guardian     Trump is making China – not America – gr
  [3]  insufficient           insufficient   guardian     Starwatch: Venus and Saturn will be visi
  [4]  direct_refute          insufficient   factcheck    Is the Great Wall of China Visible from 
        rationale: The Great Wall of China is not mentioned as being visible from space in the Guardian artic
  [5]  insufficient           insufficient   livewiki     Wikipedia: Great Wall of China
        rationale: Wikipedia article does not

In [8]:
# Claim 3 — NOISY: retriever will pull topically related but not directly relevant sources
# Expect at least 1 correlated_context classification
result_noisy = run_claim(
    "Henri Christophe built a palace in Milot.",
    label="noisy/mixed",
)


[NOISY/MIXED] Henri Christophe built a palace in Milot.
  Retrieved : 2 sources
  NLI done  : 2 sources sent to LLM
[LLM] Generating... (prompt=485 tokens, max_new=256)
[LLM] Generation complete.

  i    llm_class              nli_hint       type         title
  ---------------------------------------------------------------------------
  [1]  direct_support         refutes        livewiki     Wikipedia: Sans-Souci Palace
        rationale: The snippet clearly states that the Palace of Sans-Souci, which is located in Milot, was t
  [2]  insufficient           insufficient   livewiki     Wikipedia: Kingdom of Haiti
        rationale: The snippet provides historical context about Henri Christophe's rule but does not mention

  verdict     : supported
  confidence  : 0.90
  best source : [1]
  explanation : The Palace of Sans-Souci, located in Milot, was indeed the principal royal residence of Henri Christophe, as stated in the Wikipedia article.


## JSON Validity Check

In [ ]:
results = [
    ("supported", result_supported),
    ("refuted",   result_refuted),
    ("noisy",     result_noisy),
]

print("JSON validity check:")
all_valid = True
for label, r in results:
    if r is None:
        print(f"  [{label}] SKIP — no sources retrieved")
        continue
    valid = (
        isinstance(r.overall_verdict, str)
        and r.overall_verdict in {"supported", "refuted", "insufficient", "mixed"}
        and isinstance(r.confidence, float)
        and 0.0 <= r.confidence <= 1.0
        and isinstance(r.sources, list)
        and len(r.sources) > 0
    )
    status = "PASS" if valid else "FAIL"
    if not valid:
        all_valid = False
    print(f"  [{label}] {status}  verdict={r.overall_verdict}  conf={r.confidence:.2f}  sources={len(r.sources)}")

assert all_valid, "One or more results failed JSON validity check."
print("\n[PASS] All results are valid LLMResult objects.")

JSON validity check:
  [supported] PASS  verdict=supported  conf=0.90  sources=2
  [refuted] PASS  verdict=refuted  conf=0.90  sources=2
  [noisy] PASS  verdict=supported  conf=0.90  sources=2

[PASS] All results are valid LLMResult objects.


## Correlation Separation Check

The noisy claim (Henri Christophe / Milot) should produce at least one `correlated_context`
classification — proving the LLM distinguishes topically related but non-probative sources
from genuinely irrelevant ones.

In [10]:
if result_noisy is not None:
    classifications = [sc.classification for sc in result_noisy.sources]
    correlated_count = classifications.count("correlated_context")
    print(f"Noisy claim classifications: {classifications}")
    print(f"correlated_context count   : {correlated_count}")

    # Soft check — log a warning rather than hard fail (retrieval is non-deterministic)
    if correlated_count == 0:
        print("[WARN] No correlated_context found — retrieval may have returned weak sources.")
        print("       This is acceptable if all retrieved snippets were genuinely insufficient.")
    else:
        print(f"[PASS] {correlated_count} correlated_context classification(s) found.")
else:
    print("[SKIP] Noisy claim had no sources.")

Noisy claim classifications: ['direct_support', 'insufficient']
correlated_context count   : 0
[WARN] No correlated_context found — retrieval may have returned weak sources.
       This is acceptable if all retrieved snippets were genuinely insufficient.


## Production Model Check (Llama 3.1 8B)

Runs one claim through the full pipeline using `LLM_PROD_MODEL_NAME` regardless of the active tier.
Skip this cell during day-to-day development — run it when you want a final quality check.

In [ ]:
import importlib
import backend.app.models.llm_model as _llm_mod

# Load Llama separately — does NOT replace the dev singleton
print(f"Loading production model: {LLM_PROD_MODEL_NAME}")
print("(This takes ~2-3 min on MPS first time, faster from cache)")
llm_prod = LLMModel(model_name=LLM_PROD_MODEL_NAME, device=LLM_DEVICE, fallback_model_name=LLM_FALLBACK_MODEL_NAME)

PROD_CLAIM = "The Eiffel Tower is located in Paris, France."

print(f"\n{'='*70}")
print(f"[PROD] {PROD_CLAIM}")
print(f"{'='*70}")

sources = retrieval_svc.retrieve(PROD_CLAIM, max_results=10, use_cache=False)
classified, _nli = stance_svc.classify(PROD_CLAIM, sources)
llm_input = classified[:LLM_MAX_INPUT_SOURCES]
print(f"  Sources sent to Llama: {len(llm_input)}")

result_prod = llm_prod.classify(PROD_CLAIM, llm_input)

class_by_idx = {sc.index: sc for sc in result_prod.sources}
print(f"\n  {'i':<4} {'llm_class':<22} {'nli_hint':<14} {'type':<12} title")
print(f"  {'-'*75}")
for i, src in enumerate(llm_input, start=1):
    sc = class_by_idx.get(i)
    cls = sc.classification if sc else "insufficient"
    nli = src.stance_hint or "none"
    print(f"  [{i}]  {cls:<22} {nli:<14} {src.source_type:<12} {src.title[:40]}")
    if sc and sc.rationale:
        print(f"        rationale: {sc.rationale[:90]}")

print(f"\n  verdict     : {result_prod.overall_verdict}")
print(f"  confidence  : {result_prod.confidence:.2f}")
print(f"  best source : [{result_prod.best_source_index}]")
print(f"  explanation : {result_prod.short_explanation}")

# Compare dev vs prod verdict
if result_supported is not None:
    match = result_supported.overall_verdict == result_prod.overall_verdict
    print(f"\n  Dev verdict  : {result_supported.overall_verdict} (conf={result_supported.confidence:.2f})")
    print(f"  Prod verdict : {result_prod.overall_verdict} (conf={result_prod.confidence:.2f})")
    print(f"  Match        : {'✓ Same verdict' if match else '✗ Different verdict'}")

## Groq API Check (cloud — instant inference)

Uses the Groq free-tier API instead of a local model.  
Requires `GROQ_API_KEY` in your `.env` file — get a free key at https://console.groq.com  

**Default model**: `llama-3.1-8b-instant` (~1-2s per call, Llama 3.1 8B quality)  
**Better model**: `llama-3.3-70b-versatile` (70B, still free, best quality)

Skip this section if you don't have a Groq API key yet.

In [7]:
from backend.app.models.llm_model import GroqLLMModel, get_groq_llm_model
from backend.app.utils.constants import GROQ_MODEL_NAME, GROQ_PROD_MODEL_NAME

# Load Groq client — no model weights, just API key validation (~instant)
# GROQ_API_KEY is read automatically from .env
try:
    groq_llm = get_groq_llm_model(model_name=GROQ_MODEL_NAME)
    print(f"[Groq] Ready. Model: {groq_llm.model_name}")
    print(f"[Groq] Prod model available: {GROQ_PROD_MODEL_NAME}")
    GROQ_AVAILABLE = True
except ValueError as e:
    print(f"[Groq] Skipping — {e}")
    GROQ_AVAILABLE = False

[Groq] Model: llama-3.1-8b-instant
[Groq] Ready. Model: llama-3.1-8b-instant
[Groq] Prod model available: llama-3.3-70b-versatile


In [ ]:
if GROQ_AVAILABLE:
    GROQ_CLAIM = "The Eiffel Tower is located in Paris, France."

    # Retrieve + NLI (reuse existing service objects from earlier cells)
    groq_sources = retrieval_svc.retrieve(GROQ_CLAIM, max_results=10, use_cache=False)
    groq_classified, _groq_nli = stance_svc.classify(GROQ_CLAIM, groq_sources)
    groq_input = groq_classified[:LLM_MAX_INPUT_SOURCES]

    print(f"Sources sent to Groq: {len(groq_input)}")
    result_groq = groq_llm.classify(GROQ_CLAIM, groq_input)

    print("\n--- Per-source (Groq) ---")
    for sc in result_groq.sources:
        src = groq_input[sc.index - 1] if sc.index <= len(groq_input) else None
        title = src.title[:50] if src else "?"
        nli = src.stance_hint or "none" if src else "?"
        print(f"  [{sc.index}] {sc.classification:<22} | nli={nli:<12} | {title}")
        print(f"        rationale: {sc.rationale[:90]}")

    print(f"\n  verdict     : {result_groq.overall_verdict}")
    print(f"  confidence  : {result_groq.confidence:.2f}")
    print(f"  best source : [{result_groq.best_source_index}]")
    print(f"  explanation : {result_groq.short_explanation}")

    # Validity checks
    assert result_groq.overall_verdict in {"supported", "refuted", "insufficient", "mixed"}, "Bad verdict"
    assert 0.0 <= result_groq.confidence <= 1.0, "Confidence out of range"
    print("\n[PASS] Groq response is valid.")
else:
    print("[SKIP] Groq not available — add GROQ_API_KEY to .env to enable.")

## Final Checklist

Before moving to Phase 9 (graph building), confirm:

- [ ] Active model tier printed correctly in setup cell (DEV or PROD)
- [ ] System prompt loads from `backend/app/prompts/classify_sources_v1.txt`
- [ ] `get_llm_model()` singleton works — re-running load cell is instant
- [ ] `classify()` returns a valid `LLMResult` for all 3 sanity claims
- [ ] `overall_verdict` is one of: `supported / refuted / insufficient / mixed`
- [ ] `confidence` is a float in [0, 1]
- [ ] Per-source `classification` is one of the 4 valid labels
- [ ] Each source has a non-empty `rationale`
- [ ] LLM correctly overrides wrong NLI hints (e.g. noisy sources marked insufficient)
- [ ] `correlated_context` label appears in noisy/mixed claims
- [ ] JSON parse retry logic doesn't trigger on well-formed model output
- [ ] `--mode llm` script mode runs end-to-end without errors
- [ ] `--mode llm --llm-backend groq` runs end-to-end (requires GROQ_API_KEY)
- [ ] Groq response passes validity checks (verdict, confidence range)

**To switch to production model for final runs:**
```python
# In constants.py, change:
LLM_MODEL_NAME = LLM_PROD_MODEL_NAME  # meta-llama/Llama-3.1-8B-Instruct
```

**To use Groq 70B for highest quality:**
```python
# In constants.py, change:
GROQ_MODEL_NAME = GROQ_PROD_MODEL_NAME  # llama-3.3-70b-versatile
```